In [0]:
# ============================================================
# ANDINASTORE ANALYTICS
# seguridad/02_grants_seguridad
#
# Politica de seguridad Unity Catalog - DEV
# Implementacion mediante PySpark + spark.sql()
# ============================================================

# ============================================================
# 1. CONFIGURACION
# ============================================================

CATALOG = "andinastore_dev"
EXTERNAL_LOCATION = "ext-andinastore-raw"

GROUP_ADMIN = "andinastore_admins"
GROUP_DEVELOPER = "andinastore_developers"
GROUP_ETL = "andinastore_etl"
GROUP_ANALYST = "andinastore_analysts"

SCHEMAS = [
    "bronze",
    "silver",
    "gold"
]

TECHNICAL_GROUPS = [
    GROUP_ADMIN,
    GROUP_DEVELOPER,
    GROUP_ETL
]

ALL_GROUPS = [
    GROUP_ADMIN,
    GROUP_DEVELOPER,
    GROUP_ETL,
    GROUP_ANALYST
]

RAW_READ_GROUPS = [
    GROUP_DEVELOPER,
    GROUP_ETL
]


# ============================================================
# 2. CONTROL DE EJECUCION
# ============================================================
#
# IMPORTANTE:
#
# Los permisos DEV ya fueron aplicados y validados mediante:
#
# proceso/04_grants
#
# Por seguridad, este notebook NO vuelve a ejecutar los GRANT
# automáticamente.
#
# Cambiar a True solamente cuando sea necesario reconstruir
# explícitamente la política de permisos.
# ============================================================

APPLY_GRANTS = False


print("=" * 80)
print("ANDINASTORE - POLITICA DE SEGURIDAD UNITY CATALOG")
print("=" * 80)

print(f"\nCatalogo:          {CATALOG}")
print(f"External Location: {EXTERNAL_LOCATION}")
print(f"Aplicar GRANT:     {APPLY_GRANTS}")


# ============================================================
# 3. DEFINICION DE LA POLITICA
# ============================================================

grant_statements = []


# ============================================================
# 3.1 USE CATALOG
# Todos los roles necesitan utilizar el catalogo.
# ============================================================

for group in ALL_GROUPS:

    grant_statements.append(
        f"""
        GRANT USE CATALOG
        ON CATALOG {CATALOG}
        TO `{group}`
        """
    )


# ============================================================
# 3.2 CREATE SCHEMA
#
# Solamente administradores pueden crear nuevos schemas.
# Developers, ETL y Analysts trabajan dentro de la
# arquitectura Bronze/Silver/Gold existente.
# ============================================================

grant_statements.append(
    f"""
    GRANT CREATE SCHEMA
    ON CATALOG {CATALOG}
    TO `{GROUP_ADMIN}`
    """
)


# ============================================================
# 3.3 PERMISOS TECNICOS SOBRE BRONZE / SILVER / GOLD
#
# Admin, Developers y ETL reciben:
#
# USE SCHEMA
# SELECT
# MODIFY
# CREATE TABLE
#
# sobre las tres capas Medallion.
# ============================================================

for group in TECHNICAL_GROUPS:

    for schema in SCHEMAS:

        grant_statements.append(
            f"""
            GRANT USE SCHEMA
            ON SCHEMA {CATALOG}.{schema}
            TO `{group}`
            """
        )

        grant_statements.append(
            f"""
            GRANT SELECT
            ON SCHEMA {CATALOG}.{schema}
            TO `{group}`
            """
        )

        grant_statements.append(
            f"""
            GRANT MODIFY
            ON SCHEMA {CATALOG}.{schema}
            TO `{group}`
            """
        )

        grant_statements.append(
            f"""
            GRANT CREATE TABLE
            ON SCHEMA {CATALOG}.{schema}
            TO `{group}`
            """
        )


# ============================================================
# 3.4 PERMISOS ANALYTICS
#
# Analysts solamente consume Gold.
#
# No recibe:
# - acceso Bronze
# - acceso Silver
# - MODIFY
# - CREATE TABLE
# ============================================================

grant_statements.append(
    f"""
    GRANT USE SCHEMA
    ON SCHEMA {CATALOG}.gold
    TO `{GROUP_ANALYST}`
    """
)

grant_statements.append(
    f"""
    GRANT SELECT
    ON SCHEMA {CATALOG}.gold
    TO `{GROUP_ANALYST}`
    """
)


# ============================================================
# 3.5 RAW EXTERNAL LOCATION
#
# Developers y ETL pueden LEER RAW.
#
# Deliberadamente NO se concede:
# - WRITE FILES
# - CREATE EXTERNAL TABLE
#
# RAW debe mantenerse como fuente protegida.
# ============================================================

for group in RAW_READ_GROUPS:

    grant_statements.append(
        f"""
        GRANT READ FILES
        ON EXTERNAL LOCATION `{EXTERNAL_LOCATION}`
        TO `{group}`
        """
    )


# ============================================================
# 4. MOSTRAR POLITICA
# ============================================================

print("\n" + "=" * 80)
print("GRANTS DEFINIDOS EN LA POLITICA")
print("=" * 80)

for index, statement in enumerate(grant_statements, start=1):

    clean_statement = " ".join(statement.split())

    print(f"{index:02d}. {clean_statement}")


print("\nTotal GRANTS definidos:", len(grant_statements))


# ============================================================
# 5. EJECUCION CONTROLADA
# ============================================================
#
# APPLY_GRANTS=False
#     Solo documenta/muestra la politica.
#
# APPLY_GRANTS=True
#     Ejecuta los GRANT mediante spark.sql().
#
# Actualmente DEV ya esta configurado,
# por lo tanto debe mantenerse en False.
# ============================================================

if APPLY_GRANTS:

    print("\n" + "=" * 80)
    print("APLICANDO POLITICA DE SEGURIDAD")
    print("=" * 80)

    for index, statement in enumerate(grant_statements, start=1):

        spark.sql(statement)

        print(
            f"[OK] GRANT {index}/{len(grant_statements)} aplicado"
        )

    print("\nPolitica de seguridad aplicada correctamente.")

else:

    print("\n" + "=" * 80)
    print("MODO DOCUMENTACION / SEGURIDAD")
    print("=" * 80)

    print(
        "No se ejecutaron GRANTS porque APPLY_GRANTS=False."
    )

    print(
        "Los permisos DEV existentes permanecen sin cambios."
    )


# ============================================================
# 6. RESUMEN DE SEGURIDAD
# ============================================================

print("\n" + "=" * 80)
print("RESUMEN DE POLITICA DEV")
print("=" * 80)

print("""
ADMIN
-----
USE CATALOG
CREATE SCHEMA
USE SCHEMA Bronze/Silver/Gold
SELECT Bronze/Silver/Gold
MODIFY Bronze/Silver/Gold
CREATE TABLE Bronze/Silver/Gold


DEVELOPER
---------
USE CATALOG
USE SCHEMA Bronze/Silver/Gold
SELECT Bronze/Silver/Gold
MODIFY Bronze/Silver/Gold
CREATE TABLE Bronze/Silver/Gold
READ FILES RAW


ETL
---
USE CATALOG
USE SCHEMA Bronze/Silver/Gold
SELECT Bronze/Silver/Gold
MODIFY Bronze/Silver/Gold
CREATE TABLE Bronze/Silver/Gold
READ FILES RAW


ANALYST
-------
USE CATALOG
USE SCHEMA Gold
SELECT Gold


PRIVILEGIOS DELIBERADAMENTE NO OTORGADOS
-----------------------------------------
ALL PRIVILEGES
WRITE FILES RAW
CREATE EXTERNAL TABLE
Storage Credential administration
Ownership changes
""")

print("=" * 80)
print("FIN - POLITICA DE SEGURIDAD DEV")
print("=" * 80)